# ESCOPE: indicios visuales de lente gravitacional

Este piloto analiza cutouts VIS seleccionados manualmente en ESCOPE y ejemplos
catalogados como candidatos de grado A/B. Genera un **indice exploratorio de
evidencia visual (0-100)** y marca sobre cada imagen las regiones que lo
motivan. No es una probabilidad de lente ni una validacion fisica: los JPG
`gz_arcsinh_vis_only` no conservan la fotometria ni el mapa de ruido necesarios
para ajustar y comparar modelos de lente con rigor.

El indice usa indicios inspirados por las lectures: estructuras alargadas
tangenciales a un centro supuesto, extension angular, concentracion en un
intervalo radial y, cuando hay varias, concordancia geometrica entre ellas.
Puede fallar ante arcos tenues, lentes descentradas, estrellas, anillos no
gravitacionales o artefactos. **Unknown no significa no-lente**. Las etiquetas
A/B solo se usan para revisar el comportamiento, nunca para calcular el indice.

## Preparacion
Ejecutar en Colab con los catalogos y cutouts montados en Google Drive.
`opencv-python-headless` se usa para proponer regiones, no para confirmar
objetos. No se realiza ningun ajuste de masa ni reconstruccion de fuente.

In [ ]:
%pip install -q opencv-python-headless pandas pyarrow matplotlib

In [ ]:
from pathlib import Path
import json
import math
import os

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from google.colab import drive
except ImportError:
    drive = None

MOUNT_GOOGLE_DRIVE = drive is not None
DRIVE_MOUNT_POINT = Path('/content/drive')
if MOUNT_GOOGLE_DRIVE and not os.path.ismount(DRIVE_MOUNT_POINT):
    if DRIVE_MOUNT_POINT.exists() and any(DRIVE_MOUNT_POINT.iterdir()):
        raise RuntimeError('El punto de montaje contiene archivos; usa un runtime limpio.')
    drive.mount(str(DRIVE_MOUNT_POINT))

print('OpenCV:', cv2.__version__)

## Seleccion de objetos
`PAPER_SELECTED_OBJECTS` recoge F1--F17, P1--P3 y los nueve cutouts
adicionales de la revision v3.2 de `main_draft.tex`. F13 aparece tambien
entre los diez objetos v3.2, pero se incluye una sola vez. Los V32 son casos
para revision, algunos descartados visualmente; no son negativos confirmados.
Puedes agregar `MANUAL_OBJECT_IDS`, un CSV de ESCOPE con `object_id`, o JPG.
Los ejemplos A/B se eligen una vez con semilla fija del catalogo de lentes.
Se usa el cutout morfologico completo para *todos* los casos; no los thumbnails
ampliados del tooltip de ESCOPE.

In [ ]:
CATALOGUE_ROOT = Path('/content/drive/MyDrive/catalogues')
MORPHOLOGY_PARQUET = CATALOGUE_ROOT / 'morphology_catalogue/representations_pca_40.parquet'
LENS_CATALOGUE_CSV = CATALOGUE_ROOT / 'strong_lensing_catalogue/q1_discovery_engine_lens_catalog.csv'
CUTOUT_BASE = CATALOGUE_ROOT / 'morphology_catalogue/cutouts_jpg_gz_arcsinh_vis_only'
OUTPUT_DIR = CATALOGUE_ROOT / 'morphology_catalogue/escope_plausibility_pilot'

PAPER_SELECTED_OBJECTS = {
    'F1': 'Q1_R1_102158276_2720324240644577707',
    'F2': 'Q1_R1_102046761_-533907276258820868',
    'F3': 'Q1_R1_102021493_-585758315477722006',
    'F4': 'Q1_R1_102023478_-630909357458111475',
    'F5': 'Q1_R1_102042916_-535530619292528684',
    'F6': 'Q1_R1_102158271_2664433469646141579',
    'F7': 'Q1_R1_102045467_-529686114270426627',
    'F8': 'Q1_R1_102159186_2624781060658776321',
    'F9': 'Q1_R1_102022974_-613186503464741356',
    'F10': 'Q1_R1_102042914_-520789072292464605',
    'F11': 'Q1_R1_102018665_-575754654509316213',
    'F12': 'Q1_R1_102022477_-608978118469786580',
    'F13': 'Q1_R1_102021984_-608794490472715426',
    'F14': 'Q1_R1_102160336_2693300336680406276',
    'F15': 'Q1_R1_102158895_2741964543657441338',
    'F16': 'Q1_R1_102020063_-625302350493849139',
    'F17': 'Q1_R1_102160337_2699565276679232681',
    'P1': 'Q1_R1_102045465_-519015948268454323',
    'P2': 'Q1_R1_102160062_2738977437675467569',
    'P3': 'Q1_R1_102158275_2712359046643460193',
    'V32-01': '-638947007456847929',
    'V32-02': '2753631225658400412',
    'V32-03': '-565713145489279265',
    'V32-04': '2726412691683713923',
    'V32-05': '-568151748493588158',
    'V32-06': '-662123160468486650',
    'V32-07': '2658669444637164781',
    'V32-08': '-616166426489219165',
    'V32-09': '-636603751491594778',
}
MANUAL_OBJECT_IDS = []  # Objetos ESCOPE adicionales no incluidos en el paper
SELECTED_CSV_PATH = None  # Path('/content/drive/MyDrive/selected_escope_objects.csv')
MANUAL_IMAGE_PATHS = []   # JPG adicionales, si no aparecen en el parquet
ARTIFACT_CONTROL_IMAGE_PATHS = []  # JPG de rayas/destellos revisados visualmente
N_REFERENCE_PER_GRADE = 4
REFERENCE_SEED = 42
DISPLAY_LIMIT = 40

# Si el centro estimado falla, anota coordenadas (x, y) en pixeles de la imagen
# original. La puntuacion depende de este centro: revisalo siempre en el overlay.
CENTER_OVERRIDES = {}  # {'object_id_o_id_str': (x, y)}


def canonical_id(value):
    text = str(value).strip()
    return text[:-2] if text.endswith('.0') else text


def morphology_path(row):
    parts = str(row['id_str']).strip().split('_')
    if len(parts) < 2:
        return None
    tile, file_id = parts[-2:]
    if file_id.startswith('-'):
        file_id = f'NEG{file_id[1:]}'
    path = CUTOUT_BASE / tile / f'{tile}_{file_id}_gz_arcsinh_vis_only.jpg'
    return path if path.is_file() else None


def build_cases():
    if not MORPHOLOGY_PARQUET.is_file():
        raise FileNotFoundError(MORPHOLOGY_PARQUET)
    if not LENS_CATALOGUE_CSV.is_file():
        raise FileNotFoundError(LENS_CATALOGUE_CSV)

    morph = pd.read_parquet(MORPHOLOGY_PARQUET, columns=['id_str'])
    morph['id_str'] = morph['id_str'].astype('string').str.strip()
    morph = morph.dropna(subset=['id_str'])
    morph['object_id'] = morph['id_str'].str.rsplit('_', n=1).str[-1].map(canonical_id)
    morph = morph.dropna(subset=['object_id']).drop_duplicates('object_id')
    lens = pd.read_csv(LENS_CATALOGUE_CSV, dtype={'object_id': 'string'})
    lens['object_id'] = lens['object_id'].astype('string').map(canonical_id)
    lens['grade'] = lens['grade'].astype('string').str.strip().str.upper()
    lens = lens[lens['grade'].isin(['A', 'B'])].drop_duplicates('object_id')
    morph = morph.merge(lens[['object_id', 'grade']], on='object_id', how='left')
    references = morph[morph['grade'].isin(['A', 'B'])]

    manual_ids = [canonical_id(value) for value in MANUAL_OBJECT_IDS]
    if SELECTED_CSV_PATH is not None:
        selected = pd.read_csv(SELECTED_CSV_PATH, dtype={'object_id': 'string'})
        if 'object_id' not in selected:
            raise ValueError('El CSV de ESCOPE necesita la columna object_id.')
        manual_ids.extend(canonical_id(value) for value in selected['object_id'].dropna())

    by_id = morph.set_index('object_id', drop=False)
    by_name = morph.set_index('id_str', drop=False)
    cases, missing, seen = [], [], set()

    def add(row, cohort, paper_label=''):
        object_id = str(row['object_id'])
        if object_id in seen:
            return False
        path = morphology_path(row)
        if path is None:
            missing.append(object_id)
            return False
        seen.add(object_id)
        cases.append({
            'object_id': object_id,
            'id_str': str(row['id_str']),
            'grade': str(row.get('grade', '')) if pd.notna(row.get('grade', '')) else '',
            'cohort': cohort, 'paper_label': paper_label,
            'path': path,
        })
        return True

    for paper_label, value in PAPER_SELECTED_OBJECTS.items():
        rows = by_name if value in by_name.index else by_id
        if value not in rows.index:
            missing.append(f'{paper_label}: {value}')
            continue
        row = rows.loc[value]
        if isinstance(row, pd.DataFrame):
            row = row.iloc[0]
        cohort = 'Paper v3.2 visual triage' if paper_label.startswith('V32') else 'Paper follow-up'
        add(row, cohort, paper_label)

    for value in dict.fromkeys(manual_ids):
        rows = by_name if value in by_name.index else by_id
        if value not in rows.index:
            missing.append(value)
            continue
        row = rows.loc[value]
        if isinstance(row, pd.DataFrame):
            row = row.iloc[0]
        add(row, 'ESCOPE')

    for grade in ('A', 'B'):
        group = references[references['grade'] == grade]
        for _, row in group.sample(frac=1, random_state=REFERENCE_SEED).iterrows():
            if sum(case['cohort'] == f'Grade {grade}' for case in cases) >= N_REFERENCE_PER_GRADE:
                break
            add(row, f'Grade {grade}')

    for raw_path in MANUAL_IMAGE_PATHS:
        path = Path(raw_path)
        if not path.is_file():
            missing.append(str(path))
            continue
        case_id = path.stem
        if case_id not in seen:
            seen.add(case_id)
            cases.append({'object_id': case_id, 'id_str': case_id, 'grade': '',
                          'cohort': 'Manual JPG', 'paper_label': '', 'path': path})

    for raw_path in ARTIFACT_CONTROL_IMAGE_PATHS:
        path = Path(raw_path)
        if not path.is_file():
            missing.append(str(path))
            continue
        case_id = path.stem
        if case_id not in seen:
            seen.add(case_id)
            cases.append({'object_id': case_id, 'id_str': case_id, 'grade': '',
                          'cohort': 'Artifact control', 'paper_label': '', 'path': path})

    if missing:
        print('No localizados:', missing[:30])
    if not cases:
        raise RuntimeError('No hay cutouts para analizar. Revisa rutas e identificadores.')
    return cases


CASES = build_cases()
pd.DataFrame([{key: str(value) for key, value in case.items()} for case in CASES])

## Detector interpretable de regiones
El centro amarillo es una hipotesis de lente. El rojo marca componentes
con extension angular, elongacion y orientacion tangencial compatibles con
un arco; el naranja marca lineas largas que tocan el borde y se excluyen.
Una marca roja es una *propuesta de revision*, no una deteccion validada.

La motivacion fisica es la ecuacion de lente
$\boldsymbol{\beta}=\boldsymbol{\theta}-\boldsymbol{\alpha}(\boldsymbol{\theta})$:
imagenes multiples de una misma fuente deberian ser compatibles con una
misma posicion en el plano fuente bajo un modelo de masa. Este piloto **no
comprueba esa ecuacion**. Solo mide cinco proxies geometricos observables:
orientacion tangencial, elongacion, extension angular, coherencia radial y
contraste local. El indice suma hasta 65 puntos por la mejor region, 20 por
una segunda y 15 por concordancia entre dos regiones. El circulo cian es la
distancia radial mediana de las marcas; **no** es un radio de Einstein medido.

In [ ]:
DETECTOR_CONFIG = {
    'inner_radius_fraction': 0.09,
    'outer_radius_fraction': 0.46,
    'residual_percentile': 91.0,
    'min_tangential_alignment': 0.55,
    'min_elongation': 1.25,
    'min_angular_span_deg': 10.0,
    'max_radial_cv': 0.35,
    'long_line_fraction': 0.50,
}


def unit_interval(value):
    return float(np.clip(value, 0.0, 1.0))


def angular_span_deg(angles):
    if len(angles) < 2:
        return 0.0
    ordered = np.sort(np.mod(angles, 2.0 * np.pi))
    gaps = np.diff(np.r_[ordered, ordered[0] + 2.0 * np.pi])
    return float(np.degrees(2.0 * np.pi - gaps.max()))


def normalize_gray(rgb):
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY).astype(np.float32)
    low, high = np.percentile(gray, [1.0, 99.5])
    if high - low < 4.0:
        return None
    return np.clip((gray - low) / (high - low), 0.0, 1.0)


def estimate_center(gray):
    height, width = gray.shape
    scale = min(height, width)
    ys, xs = np.mgrid[:height, :width]
    cx, cy = (width - 1) / 2.0, (height - 1) / 2.0
    prior = np.exp(-((xs - cx) ** 2 + (ys - cy) ** 2) / (2 * (0.18 * scale) ** 2))
    smooth = cv2.GaussianBlur(gray, (0, 0), sigmaX=max(1.5, 0.022 * scale))
    peak_y, peak_x = np.unravel_index(np.argmax(smooth * prior), gray.shape)
    local = (xs - peak_x) ** 2 + (ys - peak_y) ** 2 < (0.08 * scale) ** 2
    weights = np.maximum(smooth - np.percentile(smooth[local], 35), 0.0) * local
    total = float(weights.sum())
    if total <= 0:
        return float(peak_x), float(peak_y)
    return float((weights * xs).sum() / total), float((weights * ys).sum() / total)


def long_border_lines(gray, cfg):
    height, width = gray.shape
    scale = min(height, width)
    edges = cv2.Canny((gray * 255).astype(np.uint8), 70, 150)
    lines = cv2.HoughLinesP(
        edges, 1, np.pi / 180, threshold=max(12, int(0.13 * scale)),
        minLineLength=int(cfg['long_line_fraction'] * scale),
        maxLineGap=max(2, int(0.06 * scale)),
    )
    output = []
    if lines is None:
        return output
    margin = 0.07 * scale
    # OpenCV may return (N, 1, 4), (N, 4), or (4,) depending on build/version.
    for x1, y1, x2, y2 in np.asarray(lines).reshape(-1, 4):
        near_border = lambda x, y: min(x, y, width - 1 - x, height - 1 - y) < margin
        length = math.hypot(x2 - x1, y2 - y1)
        border_supported = (near_border(x1, y1) and near_border(x2, y2)) or (
            length >= 0.68 * scale and (near_border(x1, y1) or near_border(x2, y2))
        )
        if not border_supported:
            continue
        normal_x, normal_y = -(y2 - y1) / length, (x2 - x1) / length
        t = np.linspace(0.1, 0.9, max(24, int(length)))
        sample_x = x1 + t * (x2 - x1)
        sample_y = y1 + t * (y2 - y1)

        def sample(offset):
            xi = np.clip(np.rint(sample_x + offset * normal_x).astype(int), 0, width - 1)
            yi = np.clip(np.rint(sample_y + offset * normal_y).astype(int), 0, height - 1)
            return gray[yi, xi]

        ridge = np.maximum.reduce([sample(-1), sample(0), sample(1)])
        contrast = ridge - 0.5 * (sample(-5) + sample(5))
        if np.median(contrast) > 0.18 and np.mean(contrast > 0.08) > 0.75:
            output.append((int(x1), int(y1), int(x2), int(y2)))
    return sorted(output, key=lambda row: math.hypot(row[2] - row[0], row[3] - row[1]),
                  reverse=True)[:12]


long_border_lines.hough_shape_compatible = True


def measure_component(points_xy, residual, center, threshold, cfg, shape):
    height, width = shape
    scale = min(height, width)
    pts = points_xy.astype(np.float64)
    centroid = pts.mean(axis=0)
    centered = pts - centroid
    eigenvalues, eigenvectors = np.linalg.eigh(np.cov(centered, rowvar=False))
    eigenvalues = np.maximum(eigenvalues, 1e-6)
    major = eigenvectors[:, np.argmax(eigenvalues)]
    elongation = float(np.sqrt(eigenvalues.max() / eigenvalues.min()))
    radial = centroid - np.asarray(center)
    radius = float(np.linalg.norm(radial))
    tangent = np.array([-radial[1], radial[0]]) / max(radius, 1e-6)
    alignment = float(abs(np.dot(major, tangent)))
    offsets = pts - np.asarray(center)
    radii = np.linalg.norm(offsets, axis=1)
    span = angular_span_deg(np.arctan2(offsets[:, 1], offsets[:, 0]))
    radial_cv = float(np.std(radii) / max(float(np.mean(radii)), 1e-6))
    values = residual[points_xy[:, 1], points_xy[:, 0]]
    strength = float(np.mean(values) / max(threshold, 0.01))
    touches_border = bool(np.any(
        (pts[:, 0] <= 1) | (pts[:, 1] <= 1)
        | (pts[:, 0] >= width - 2) | (pts[:, 1] >= height - 2)
    ))

    features = {
        'tangential': unit_interval((alignment - 0.45) / 0.55),
        'elongation': unit_interval((elongation - 1.0) / 2.5),
        'angular_span': unit_interval((span - 8.0) / 52.0),
        'radial_coherence': unit_interval(1.0 - radial_cv / 0.30),
        'contrast': unit_interval((strength - 1.0) / 2.0),
    }
    component_score = sum(features[key] * weight for key, weight in (
        ('tangential', 0.30), ('elongation', 0.20),
        ('angular_span', 0.25), ('radial_coherence', 0.15),
        ('contrast', 0.10),
    ))
    accepted = (
        cfg['inner_radius_fraction'] * scale <= radius <= cfg['outer_radius_fraction'] * scale
        and alignment >= cfg['min_tangential_alignment']
        and elongation >= cfg['min_elongation']
        and span >= cfg['min_angular_span_deg']
        and radial_cv <= cfg['max_radial_cv']
        and not touches_border
        and len(pts) <= 0.08 * height * width
    )
    return {
        'centroid_x': float(centroid[0]), 'centroid_y': float(centroid[1]),
        'radius_px': radius, 'angular_span_deg': span,
        'tangential_alignment': alignment, 'elongation': elongation,
        'radial_cv': radial_cv, 'contrast_ratio': strength,
        'component_score': float(component_score), 'accepted': bool(accepted),
        'features': features, 'pixels': points_xy,
    }


def pair_agreement(components, center):
    if len(components) < 2:
        return 0.0
    best = 0.0
    for i, left in enumerate(components):
        for right in components[i + 1:]:
            left_angle = math.atan2(left['centroid_y'] - center[1], left['centroid_x'] - center[0])
            right_angle = math.atan2(right['centroid_y'] - center[1], right['centroid_x'] - center[0])
            separation = abs((left_angle - right_angle + math.pi) % (2 * math.pi) - math.pi)
            radius_gap = abs(left['radius_px'] - right['radius_px']) / max(
                left['radius_px'], right['radius_px'], 1.0)
            best = max(best, unit_interval((math.degrees(separation) - 25) / 100)
                       * unit_interval(1.0 - radius_gap / 0.4))
    return best


def analyze_image(rgb, center_override=None, cfg=DETECTOR_CONFIG):
    height, width = rgb.shape[:2]
    if min(height, width) < 48:
        return {'status': 'insufficient_resolution', 'score': None, 'original': rgb}
    gray = normalize_gray(rgb)
    if gray is None:
        return {'status': 'insufficient_contrast', 'score': None, 'original': rgb}
    center = center_override if center_override is not None else estimate_center(gray)
    center = (float(center[0]), float(center[1]))
    if not (0 <= center[0] < width and 0 <= center[1] < height):
        raise ValueError(f'Centro fuera de la imagen: {center}')

    scale = min(height, width)
    yy, xx = np.mgrid[:height, :width]
    distance = np.hypot(xx - center[0], yy - center[1])
    annulus = ((distance >= cfg['inner_radius_fraction'] * scale)
               & (distance <= cfg['outer_radius_fraction'] * scale))
    smooth = cv2.GaussianBlur(gray, (0, 0), sigmaX=max(4.0, 0.055 * scale))
    residual = np.maximum(gray - smooth, 0.0)
    reference = residual[annulus]
    median = float(np.median(reference))
    noise = float(1.4826 * np.median(np.abs(reference - median)))
    threshold = max(float(np.percentile(reference, cfg['residual_percentile'])),
                    median + 2.5 * noise, 0.04)
    binary = ((residual > threshold) & annulus).astype(np.uint8)
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE,
                              cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)))
    artifact_lines = long_border_lines(gray, cfg)
    artifact_mask = np.zeros((height, width), dtype=np.uint8)
    for x1, y1, x2, y2 in artifact_lines:
        cv2.line(artifact_mask, (x1, y1), (x2, y2), 255,
                 max(2, round(0.018 * scale)), cv2.LINE_AA)
    binary[artifact_mask > 0] = 0
    count, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    min_area = max(4, round(0.0002 * height * width))
    components = []
    for index in range(1, count):
        if stats[index, cv2.CC_STAT_AREA] < min_area:
            continue
        ys, xs = np.where(labels == index)
        points = np.column_stack([xs, ys]).astype(np.int32)
        if len(points) < 3:
            continue
        components.append(measure_component(points, residual, center, threshold,
                                            cfg, (height, width)))

    accepted = [component for component in components if component['accepted']]
    accepted.sort(key=lambda item: item['component_score'], reverse=True)
    pair = pair_agreement(accepted[:6], center)
    primary = accepted[0]['component_score'] if accepted else 0.0
    secondary = accepted[1]['component_score'] if len(accepted) > 1 else 0.0
    # Fixed, inspectable weights; not fit or calibrated on A/B labels.
    contributions = {
        'best_region': 65.0 * primary,
        'second_region': 20.0 * secondary,
        'multi_region_geometry': 15.0 * pair,
    }
    score = int(round(np.clip(sum(contributions.values()), 0.0, 100.0)))

    overlay = rgb.copy().astype(np.float32)
    for rank, component in enumerate(accepted[:6], start=1):
        pixels = component['pixels']
        original = overlay[pixels[:, 1], pixels[:, 0]]
        overlay[pixels[:, 1], pixels[:, 0]] = 0.45 * original + 0.55 * np.array([255, 55, 55])
    overlay = np.clip(overlay, 0, 255).astype(np.uint8)
    for rank, component in enumerate(accepted[:6], start=1):
        cv2.putText(overlay, str(rank),
                    (int(component['centroid_x']) + 3, int(component['centroid_y']) - 3),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1, cv2.LINE_AA)
    for x1, y1, x2, y2 in artifact_lines:
        cv2.line(overlay, (x1, y1), (x2, y2), (255, 165, 0), 1, cv2.LINE_AA)
    cv2.drawMarker(overlay, tuple(round(value) for value in center), (255, 230, 0),
                   markerType=cv2.MARKER_CROSS, markerSize=11, thickness=1)
    if accepted:
        radius = round(np.median([item['radius_px'] for item in accepted[:6]]))
        cv2.circle(overlay, tuple(round(value) for value in center), radius,
                   (0, 220, 220), 1, cv2.LINE_AA)

    return {
        'status': 'exploratory_visual_only', 'score': score,
        'original': rgb, 'overlay': overlay, 'center': center,
        'components': components, 'accepted': accepted,
        'artifact_lines': artifact_lines, 'threshold': threshold,
        'contributions': contributions, 'pair_agreement': pair,
        'quality_flag': 'possible_border_line' if artifact_lines else '',
    }

## Control interno rapido
Este ejemplo sintetico comprueba que una estructura curva alrededor del centro
reciba mas apoyo que una linea recta que atraviesa toda la imagen. **No es una
validacion cientifica** ni garantiza ese orden en cutouts reales.

In [ ]:
rng = np.random.default_rng(11)
size = 160
grid_y, grid_x = np.mgrid[:size, :size]
core = 130 * np.exp(-((grid_x - 80) ** 2 + (grid_y - 80) ** 2) / (2 * 9 ** 2))
base = np.clip(15 + core + rng.normal(0, 3, (size, size)), 0, 255).astype(np.uint8)
arc = cv2.cvtColor(base, cv2.COLOR_GRAY2RGB)
line = arc.copy()
cv2.ellipse(arc, (80, 80), (42, 42), 0, 20, 130, (225, 225, 225), 3)
cv2.ellipse(arc, (80, 80), (42, 42), 0, 205, 280, (190, 190, 190), 3)
cv2.line(line, (1, 55), (158, 105), (225, 225, 225), 3)
arc_test = analyze_image(arc, center_override=(80, 80))
line_test = analyze_image(line, center_override=(80, 80))
print('Control sintetico:', {'arco': arc_test['score'], 'linea': line_test['score']})
assert arc_test['score'] > line_test['score'], 'Revisar el detector: el control sintetico fallo.'

## Analisis de los cutouts
El indice es reproducible para una imagen y centro dados. Puede variar mucho
si cambia el centro o el preprocesamiento; comprueba cada overlay antes de
interpretar la tabla. Las etiquetas A/B no entran en `analyze_image`.

In [ ]:
if not getattr(long_border_lines, 'hough_shape_compatible', False):
    raise RuntimeError(
        'La definicion del detector esta obsoleta. Ejecuta de nuevo la celda '
        'Detector interpretable de regiones antes de analizar los cutouts.'
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
results = []
details = {}
region_rows = []
for case in CASES:
    path = case['path']
    bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if bgr is None:
        results.append({**{key: case[key] for key in ('object_id', 'id_str', 'grade', 'cohort', 'paper_label')},
                        'status': 'read_error', 'score_0_100': np.nan})
        continue
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    override = CENTER_OVERRIDES.get(case['object_id'], CENTER_OVERRIDES.get(case['id_str']))
    result = analyze_image(rgb, center_override=override)
    details[case['object_id']] = result
    for index, region in enumerate(result.get('accepted', []), start=1):
        region_rows.append({
            'object_id': case['object_id'], 'region': index,
            'centroid_x': region['centroid_x'],
            'centroid_y': region['centroid_y'],
            'radius_px': region['radius_px'],
            'angular_span_deg': region['angular_span_deg'],
            'tangential_alignment': region['tangential_alignment'],
            'elongation': region['elongation'],
            'radial_cv': region['radial_cv'],
            'contrast_ratio': region['contrast_ratio'],
            'component_score': region['component_score'],
        })
    overlay_path = OUTPUT_DIR / f"{case['object_id']}_visual_evidence.png"
    if 'overlay' in result:
        cv2.imwrite(str(overlay_path), cv2.cvtColor(result['overlay'], cv2.COLOR_RGB2BGR))
    results.append({
        'object_id': case['object_id'], 'id_str': case['id_str'],
        'grade': case['grade'], 'cohort': case['cohort'],
        'paper_label': case['paper_label'],
        'score_0_100': result['score'], 'status': result['status'],
        'quality_flag': result.get('quality_flag', ''),
        'n_marked_regions': len(result.get('accepted', [])),
        'n_possible_border_lines': len(result.get('artifact_lines', [])),
        'center_x': result.get('center', (np.nan, np.nan))[0],
        'center_y': result.get('center', (np.nan, np.nan))[1],
        'best_region_points': result.get('contributions', {}).get('best_region', np.nan),
        'second_region_points': result.get('contributions', {}).get('second_region', np.nan),
        'multi_region_points': result.get('contributions', {}).get('multi_region_geometry', np.nan),
        'overlay_path': str(overlay_path) if 'overlay' in result else '',
    })

summary = pd.DataFrame(results).sort_values('score_0_100', ascending=False, na_position='last')
summary.to_csv(OUTPUT_DIR / 'visual_evidence_scores.csv', index=False)
pd.DataFrame(region_rows).to_csv(OUTPUT_DIR / 'visual_evidence_regions.csv', index=False)
with (OUTPUT_DIR / 'run_manifest.json').open('w') as handle:
    json.dump({
        'input_paths': {case['object_id']: str(case['path']) for case in CASES},
        'detector_config': DETECTOR_CONFIG,
        'manual_object_ids': MANUAL_OBJECT_IDS,
        'paper_selected_objects': PAPER_SELECTED_OBJECTS,
        'center_overrides': CENTER_OVERRIDES,
        'reference_seed': REFERENCE_SEED,
        'score_semantics': 'uncalibrated visual evidence; not P(lens|image)',
    }, handle, indent=2)
display(summary)
controls = summary[summary['cohort'] == 'Artifact control']
if not controls.empty:
    print('Controles visuales de artefactos (una puntuacion alta es un fallo que revisar):')
    display(controls[['object_id', 'score_0_100', 'n_possible_border_lines', 'quality_flag']])
print('Resultados:', OUTPUT_DIR)

## Galeria explicativa
Rojo: regiones que **contribuyeron**. Amarillo: centro supuesto. Cian:
distancia radial mediana de las regiones marcadas. Naranja: posibles lineas
largas de borde descartadas. La ausencia de rojo indica que este detector no
encontro apoyo, **no** que el objeto no sea una lente.

In [ ]:
for _, row in summary.head(DISPLAY_LIMIT).iterrows():
    result = details.get(row['object_id'])
    if result is None:
        continue
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(result['original'])
    axes[0].set_title('Cutout original')
    axes[1].imshow(result.get('overlay', result['original']))
    axes[1].set_title(f"Indicios visuales: {row['score_0_100']}/100")
    for axis in axes:
        axis.axis('off')
    fig.suptitle(f"{row['paper_label']} | {row['object_id']} | {row['cohort']}", fontsize=10)
    plt.tight_layout()
    plt.show()
    accepted = result.get('accepted', [])
    if accepted:
        measures = pd.DataFrame([{
            'region': index + 1,
            'span_deg': part['angular_span_deg'],
            'tangential_alignment': part['tangential_alignment'],
            'radius_px': part['radius_px'],
            'radial_cv': part['radial_cv'],
            'region_score': part['component_score'],
        } for index, part in enumerate(accepted[:6])])
        display(measures.round(3))

## Interpretacion y siguiente fase

- Compara los overlays de A/B con tus selecciones ESCOPE; **no** calcules
  precision ni pureza usando objetos Unknown como negativos.
- Incorpora ejemplos visualmente identificados como artefactos en
  `ARTIFACT_CONTROL_IMAGE_PATHS`. Este indice puede darles puntuaciones altas;
  eso obliga a revisar cada caso y a mejorar el detector, no a ocultar el fallo.
- Registra manualmente, en un conjunto independiente, que marcas corresponden
  a arcos reales, estrellas, galaxias vecinas o artefactos. Repite con centros
  corregidos y mide la sensibilidad del indice.
- Para un **indice de consistencia fisica**, sustituye los JPG por datos VIS
  cientificos (imagen FITS, mapa de ruido, PSF, escala de pixel, mascara). Ajusta
  modelos de lente y de no-lente con PyAutoLens/lenstronomy, compara predicciones
  en el plano imagen y comprueba la coherencia entre imagenes multiples en el
  plano fuente. Solo despues de validar con positivos y negativos independientes
  se puede calibrar una probabilidad; los grados A/B no son verdad fisica final.
- No integres el 0-100 en ESCOPE hasta contar con una evaluacion de falsos
  positivos, sensibilidad a PSF/centro y calibracion sobre datos independientes.